# <font color='#000000'>__Atividade Colaborativa__</font>
## <font color='#1c8a23'>Kloppy</font>
#### <font color='#4b4b4b'>Master Big Data Aplicado ao Futebol <br> Módulo 5 - Análise de dados no futebol com Python <br> Desenvolvido por: Hugo Alves </font>

<br>

O package [kloppy](https://kloppy.pysport.org/) foi desenvolvido pela [PySport](https://pysport.org/) e surgiu da dificuldade em unanimizar a informação dos diferentes provedores de dados no futebol. Cada empresa adota os seus fomatos, definições de conceitos, sistemas de coordenadas, etc., e torna-se difícil para qualquer pessoa construir código e soluções que não sejam feitas à medida do provedor que estão a consultar.
Com esta biblioteca, estes dados - que podem ser metadados ou eventos das partidas, dados de tracking, ou até dados de suporte para análise de vídeo - passam a poder ser carregados, filtrados, transformados e exportados de forma independente do provedor. <br>
A imagem abaixo ilustra na perfeição o objetivo (e principal função) do kloppy:
<figure style="text-align: center;">
<img src="https://kloppy.pysport.org/user-guide/getting-started/images/what-kloppy-does.png" alt="Kloppy" width="400" height="300">
<figcaption>Fonte: Kloppy</figcaption>
</figure>
<br>

Neste notebook, percorremos alguns exemplos de como carregar dados de diferentes [fornecedores de dados](https://kloppy.pysport.org/user-guide/loading-data/#supported-data-providers) (Statsbomb, Metrica Sports, e OPTA/StatsPerform), como os podemos filtrar, trabalhar e converter num formato de DataFrame para construir depois código comum, sem termos de o adaptar à fonte e formato de origem da informação. Exploramos ainda uma outra função do kloppy relativamente simples mas interessante, que permite especificar eventos ou sequências de eventos de interesse e identificar a sua ocorrência nos nossos conjuntos de dados.

# <font color="#1c8a23">__________________</font>
## <font color="#4b4b4b">Índice de Conteúdos</font> <a class="anchor" id="toc"></a>
[1. Setup Inicial](#setup)<br>

[2. Carregamento de dados](#load)<br>
- [2.1. Statsbomb](#statsbomb)<br>
- [2.2. Metrica Sports](#metrica)<br>
- [2.3. OPTA (StatsPerform)](#opta)<br>

[3. Comparação e Potencialidades](#cp)<br>

# <font color="#1c8a23">____________</font>
## <font color='#4b4b4b'>1. Setup Inicial</font> <a class="anchor" id="setup"></a>

Vamos começar por importar os packages que vamos precisar. Com exceção do kloppy, o mais provável é que os restantes packages não precisem de instalação.

In [1]:
#!pip install kloppy

In [2]:
from kloppy import statsbomb, metrica, skillcorner, opta
from kloppy import event_pattern_matching as pm

import requests
import json

import pandas as pd

# <font color="#1c8a23">_______________________</font>
## <font color='#4b4b4b'>2. Carregamento de dados</font> <a class="anchor" id="load"></a>
<br>

Podemos partir para o carregamento de dados de várias fontes para observar o output dado pelas diferentes funções deste package. Idealmente, esperamos chegar a DataFrames com formatos semelhantes entre si, e que possam servir de base para análises futuras.

### <font color='#1c8a23'>2.1. Statsbomb</font> <a class="anchor" id="statsbomb"></a>
[Regressar ao Índice](#toc)

[Documentação kloppy](https://kloppy.pysport.org/user-guide/loading-data/statsbomb/) <br>
Começando pelos dados da Statsbomb, podemos usufruir do repositório público de dados no [GitHub](https://github.com/statsbomb/open-data) para extrair os metadados da partida e os seus eventos.

In [3]:
# Este é o primeiro ID que aparece ao abrir a pasta de eventos ou lineups no GitHub. Podemos alterar para qualquer outro que esteja nestas pastas
id_match_statsbomb = 15946
url_statsbomb = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"

# Links
urls_statsbomb = {
  "events": f"{url_statsbomb}/events/{id_match_statsbomb}.json",
  "lineups": f"{url_statsbomb}/lineups/{id_match_statsbomb}.json"
}

Com estes IDs, podemos transferir os ficheiros json para os conteúdos do Colab. Este não é um passo necessário, visto que podíamos simplesmente chamar os URLs na função statsbomb.load() abaixo, mas sempre temos uma visão mais completa e, mais importante que isso, temos os ficheiros também do nosso lado.

In [4]:
for name, url in urls_statsbomb.items():
  with open(f"statsbomb_{name}.json", "w") as f:
    json.dump(requests.get(url).json(), f)

Vamos aproveitar o exemplo da documentação e extrair apenas dois tipos de eventos: passes e remates. Este parâmetro é opcional (se não especificarmos, todos os eventos são extraídos) e a lista completa de tipos de eventos encontra-se disponível [aqui](https://kloppy.pysport.org/reference/event-data/event-types/).

Nesta documentação, podemos ainda selecionar um tipo de evento específico para perceber que atributos adicionais são fornecidos (e por quais provedores de dados) ou até investigar os diferentes [qualifiers](https://kloppy.pysport.org/reference/event-data/qualifiers/), que oferecem mais detalhe sobre cada ocorrência do jogo.

In [5]:
data_statsbomb = statsbomb.load(
  event_data = "statsbomb_events.json",
  lineup_data = "statsbomb_lineups.json",
  coordinates = "statsbomb",
  event_types = ["pass", "shot"]
)
data_statsbomb

<EventDataset record_count=1191>

Nota: O código acima é o mais indicado para carregar ficheiros próprios ou de fontes não públicas. Neste caso, em que estamos a trabalhar com dados open-source da statsbomb, até podíamos simplificar o carregamento apenas com recurso ao ID do jogo, como demonstrado no código abaixo.

In [6]:
data_statsbomb_open = statsbomb.load_open_data(
  match_id = id_match_statsbomb,
  coordinates = "statsbomb",
  event_types = ["pass", "shot"]
)
data_statsbomb_open

/usr/local/lib/python3.12/dist-packages/kloppy/_providers/statsbomb.py:84: UserWarning: 

You are about to use StatsBomb public data.
By using this data, you are agreeing to the user agreement. 
The user agreement can be found here: https://github.com/statsbomb/open-data/blob/master/LICENSE.pdf

  warnings.warn(


<EventDataset record_count=1191>

Dentro deste objeto `data_statsbomb`/`data_statsbomb_open`, o kloppy até permite filtrar desde logo os dados por algum tipo de evento, resultado do evento, ou pela combinação evento-resultado. Desta forma podemos, por exemplo, partir de um cenário em que temos todos os eventos de um jogo, e criar datasets específicos para cada domínio de análise com os qualifiers associados.

In [7]:
# Filtrar por todos os passes
events_passes = data_statsbomb.find_all("pass")
# Filtrar por todos os passes falhados
events_passes_incomplete = data_statsbomb.find_all("pass.incomplete")
# Filtrar por todos os eventos que resultaram em golo (tendencialmente remates)
events_goals = data_statsbomb.find_all(".goal")
events_goals

[StatsBombShotEvent(qualifiers=[SetPieceQualifier(value=<SetPieceType.FREE_KICK: 'FREE_KICK'>), BodyPartQualifier(value=<BodyPart.LEFT_FOOT: 'LEFT_FOOT'>)]),
 StatsBombShotEvent(qualifiers=[BodyPartQualifier(value=<BodyPart.RIGHT_FOOT: 'RIGHT_FOOT'>)]),
 StatsBombShotEvent(qualifiers=[BodyPartQualifier(value=<BodyPart.LEFT_FOOT: 'LEFT_FOOT'>)])]

Os métodos `.prev()` e `.next()` também são extremamente úteis para devolver eventos que antecedem ou seguem algum acontecimento do jogo, respetivamente. Por exemplo, podemos chegar à assistência para o primeiro golo a partir do conjunto de golos que já filtrámos anteriormente.

In [8]:
events_assist_first = events_goals[0].prev("pass.complete")
events_assist_first

StatsBombPassEvent(qualifiers=[BodyPartQualifier(value=<BodyPart.RIGHT_FOOT: 'RIGHT_FOOT'>)])

Como estas, existem outras transformações potencialmente úteis ao nível dos dados de eventos, tracking, e metadados da partida. A documentação completa deste package encontra-se disponível [aqui](https://kloppy.pysport.org/user-guide/getting-started/) e apresenta, logo na sua página inicial, exemplos para grande parte destas transformações.

Por fim, podemos converter o objeto &lt;EventDataset&gt; em que temos estado a trabalhar num DataFrame, um formato potencialmente mais interessante para certos tipos de análise.

In [9]:
df_statsbomb = data_statsbomb.to_df()
df_statsbomb.head()

,event_id,event_type,period_id,timestamp,end_timestamp,ball_state,ball_owning_team,team_id,player_id,coordinates_x,...,end_coordinates_x,end_coordinates_y,receiver_player_id,set_piece_type,body_part_type,result,success,pass_type,is_under_pressure,is_counter_attack
0,549567bd-36de-4ac8-b8dc-6b5d3f1e4be8,PASS,1,0 days 00:00:00.575000,0 days 00:00:02.590669,alive,206,206,6581,60.95,...,33.75,27.95,6855,KICK_OFF,LEFT_FOOT,COMPLETE,True,None,None,None
1,4e4e4cad-9897-43ec-842d-585a4077f6ce,PASS,1,0 days 00:00:03.864000,0 days 00:00:07.151421,alive,206,206,6855,36.75,...,86.45,74.15,None,None,RIGHT_FOOT,INCOMPLETE,False,LONG_BALL,None,None
2,be27cc25-92b5-4696-b43c-aad957a6119a,PASS,1,0 days 00:00:07.152000,0 days 00:00:08.848529,alive,217,217,5203,33.55,...,35.05,18.25,5470,None,None,COMPLETE,True,HIGH_PASS,True,None
3,b33c0b7f-7456-4efe-b43c-5fd7cbd14689,PASS,1,0 days 00:00:08.848000,0 days 00:00:09.983950,alive,217,217,5470,35.05,...,36.15,5.25,5477,None,HEAD,COMPLETE,True,HEAD_PASS,None,None
4,c587e5ce-fe6e-4cfb-b510-8a8e193699d3,PASS,1,0 days 00:00:10.873000,0 days 00:00:11.630764,alive,217,217,5477,34.25,...,25.25,1.55,5211,None,RIGHT_FOOT,COMPLETE,True,None,None,None


E ainda analisar as diferentes colunas ao nível dos missing values e data types.

In [10]:
df_statsbomb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1191 entries, 0 to 1190
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   event_id            1191 non-null   object         
 1   event_type          1191 non-null   object         
 2   period_id           1191 non-null   int64          
 3   timestamp           1191 non-null   timedelta64[ns]
 4   end_timestamp       1163 non-null   timedelta64[ns]
 5   ball_state          1191 non-null   object         
 6   ball_owning_team    1191 non-null   object         
 7   team_id             1191 non-null   object         
 8   player_id           1191 non-null   object         
 9   coordinates_x       1191 non-null   float64        
 10  coordinates_y       1191 non-null   float64        
 11  end_coordinates_x   1191 non-null   float64        
 12  end_coordinates_y   1191 non-null   float64        
 13  receiver_player_id  984 non-null 

__Nota:__ Existem no kloppy outros métodos complementares e parâmetros dentro deste método `.to_df()` que permitem uma transformação mais "controlada" dos eventos em DataFrame, mas vamos aqui cingir-nos à abordagem mais simples. Sem prejuízo de investigar o resto da documentação, algumas destas formas encontram-se [aqui](https://kloppy.pysport.org/user-guide/getting-started/#to-a-polarspandas-dataframe) e [aqui](https://kloppy.pysport.org/user-guide/exporting-data/dataframes/).

### <font color='#1c8a23'>2.2. Metrica Sports</font> <a class="anchor" id="metrica"></a>
[Regressar ao Índice](#toc)

[Documentação kloppy](https://kloppy.pysport.org/user-guide/loading-data/metrica/) <br>
Tal como com a statsBomb, existe também um repositório público com dados no [GitHub](https://github.com/metrica-sports/sample-data), ao qual podemos aceder. O processo será bastante semelhante ao que seguimos no caso anterior: este é um dos grandes propósitos (e vantagens) deste package. A diferença está apenas no nome da função que vamos chamar, que será `.load_event()` ao invés da `.load()` utilizada para a Statsbomb.


In [11]:
data_metrica = metrica.load_event(
  event_data = "https://raw.githubusercontent.com/metrica-sports/sample-data/refs/heads/master/data/Sample_Game_3/Sample_Game_3_events.json",
  meta_data = "https://raw.githubusercontent.com/metrica-sports/sample-data/refs/heads/master/data/Sample_Game_3/Sample_Game_3_metadata.xml",
  coordinates = "metrica",
  event_types = ["pass", "shot"]
)
data_metrica

<EventDataset record_count=1493>

Maravilha. E podemos converter em DataFrame...

In [12]:
df_metrica = data_metrica.to_df()
df_metrica.head()

,event_id,event_type,period_id,timestamp,end_timestamp,ball_state,ball_owning_team,team_id,player_id,coordinates_x,coordinates_y,end_coordinates_x,end_coordinates_y,receiver_player_id,set_piece_type,result,success,body_part_type
0,2,PASS,1,0 days 00:00:14.400000,0 days 00:00:15.040000,alive,FIFATMA,FIFATMA,P3577,0.50125,0.48725,0.49864,0.48705,P3574,KICK_OFF,COMPLETE,True,None
1,4,PASS,1,0 days 00:00:15.320000,0 days 00:00:17,alive,FIFATMA,FIFATMA,P3574,0.49700,0.48500,0.63373,0.63449,P3575,None,COMPLETE,True,None
2,6,PASS,1,0 days 00:00:18.560000,0 days 00:00:20.240000,alive,FIFATMA,FIFATMA,P3575,0.66986,0.59707,0.80602,0.39821,P3569,None,COMPLETE,True,None
3,8,PASS,1,0 days 00:00:21.160000,0 days 00:00:23.160000,alive,FIFATMA,FIFATMA,P3569,0.80929,0.42922,0.79906,0.81522,P3570,None,COMPLETE,True,None
4,10,PASS,1,0 days 00:00:23.880000,0 days 00:00:25.080000,alive,FIFATMA,FIFATMA,P3570,0.79756,0.81998,0.68101,0.98059,P3571,None,COMPLETE,True,None


... e observar as colunas que temos, bem como os respetivos missing values e data types. A partir deste momento, já temos um DataFrame "normal" do pandas, com o qual podemos trabalhar.

In [13]:
df_metrica.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1493 entries, 0 to 1492
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   event_id            1493 non-null   object         
 1   event_type          1493 non-null   object         
 2   period_id           1493 non-null   int64          
 3   timestamp           1493 non-null   timedelta64[ns]
 4   end_timestamp       1121 non-null   timedelta64[ns]
 5   ball_state          1493 non-null   object         
 6   ball_owning_team    1493 non-null   object         
 7   team_id             1493 non-null   object         
 8   player_id           1493 non-null   object         
 9   coordinates_x       1493 non-null   float64        
 10  coordinates_y       1493 non-null   float64        
 11  end_coordinates_x   1121 non-null   float64        
 12  end_coordinates_y   1121 non-null   float64        
 13  receiver_player_id  1121 non-null

Uma novidade no caso dos dados da Metrica Sports é que podemos também acrescentar dados de tracking à equação. <br>
A lógica é semelhante à dos eventos: uma função recebe dados de um determinado provedor, e estandardiza-os para um formato comum.
Eis um exemplo simples abaixo:

In [14]:
data_metrica_tracking = metrica.load_tracking_csv(
  home_data = "https://raw.githubusercontent.com/metrica-sports/sample-data/master/data/Sample_Game_1/Sample_Game_1_RawTrackingData_Home_Team.csv",
  away_data = "https://raw.githubusercontent.com/metrica-sports/sample-data/master/data/Sample_Game_1/Sample_Game_1_RawTrackingData_Away_Team.csv",
  coordinates = "metrica"
)
data_metrica_tracking.to_df().head()

,period_id,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y,ball_z,ball_speed,home_11_x,...,home_13_d,home_13_s,away_28_x,away_28_y,away_28_d,away_28_s,home_14_x,home_14_y,home_14_d,home_14_s
0,1,0 days 00:00:00.040000,1,None,None,0.45472,0.61291,None,None,0.00082,...,None,None,NaN,NaN,None,None,NaN,NaN,None,None
1,1,0 days 00:00:00.080000,2,None,None,0.49645,0.59344,None,None,0.00096,...,None,None,NaN,NaN,None,None,NaN,NaN,None,None
2,1,0 days 00:00:00.120000,3,None,None,0.53716,0.57444,None,None,0.00114,...,None,None,NaN,NaN,None,None,NaN,NaN,None,None
3,1,0 days 00:00:00.160000,4,None,None,0.55346,0.57769,None,None,0.00121,...,None,None,NaN,NaN,None,None,NaN,NaN,None,None
4,1,0 days 00:00:00.200000,5,None,None,0.55512,0.59430,None,None,0.00129,...,None,None,NaN,NaN,None,None,NaN,NaN,None,None


In [15]:
data_metrica_tracking.to_df().shape

(145006, 121)

Como vemos, num instante chegámos a um DataFrame preparado e extremamente detalhado com as coordenadas dos 22 jogadores (e da bola) a cada 4 centésimas de segundo, resultando num total de mais de 145 mil linhas e 121 colunas.

### <font color='#1c8a23'>2.3. OPTA (StatsPerform)</font> <a class="anchor" id="opta"></a>
[Regressar ao Índice](#toc)

[Documentação kloppy](https://kloppy.pysport.org/user-guide/loading-data/statsperform/) <br>

Por fim, seria interessante explorar os dados da OPTA e observar a transformação dos dados de eventos para um formato comum, como verificámos nos dois exemplos anteriores. Contudo, dada a escassez de acesso a samples de ficheiros de dados deste provedor, tal não será possível. Abaixo fica comentado código para trazer a pasta com os ficheiros F24 que nos foram disponibilizados para este módulo e que pode ser adaptado para chamar outras pastas. O inconveniente neste caso é a necessidade de um ficheiro F7 com outro tipo de dados da partida, e ao qual não temos acesso livre. <br>
Importa ainda realçar que, como detalhado na documentação, a função `.load()` abaixo utilizada já não seria a adequada para carregar ficheiros MA1 e MA3 (um formato aparentemente mais recente da OPTA): neste caso, a função a utilizar seria a `.load_event()`.

In [16]:
#import os
#import zipfile
#from google.colab import files

In [17]:
#folder_name = "OPTA Data"

In [18]:
"""
if os.path.isdir(folder_name):
  print(f"A pasta '{folder_name}' já existe. A usar os ficheiros existentes.")
else:
  print(f"Pasta não encontrada. Necessário fazer upload do ficheiro ZIP.")

  uploaded = files.upload()
  if not uploaded:
    raise RuntimeError("Nenhum ficheiro foi carregado.")

  zip_name = list(uploaded.keys())[0]
  with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(folder_name)
"""

'\nif os.path.isdir(folder_name):\n  print(f"A pasta \'{folder_name}\' já existe. A usar os ficheiros existentes.")\nelse:\n  print(f"Pasta não encontrada. Necessário fazer upload do ficheiro ZIP.")\n\n  uploaded = files.upload()\n  if not uploaded:\n    raise RuntimeError("Nenhum ficheiro foi carregado.")\n\n  zip_name = list(uploaded.keys())[0]\n  with zipfile.ZipFile(zip_name, "r") as zip_ref:\n    zip_ref.extractall(folder_name)\n'

In [19]:
"""
data_opta = opta.load(
    f7_data = "",
    f24_data = "OPTA Data/OPTA Data/F24 - Portugal/f24-99-2020-2133169-eventdetails.xml",
    # Optional arguments
    coordinates = "opta",
    event_types = ["pass", "shot"]
)
data_opta
"""

'\ndata_opta = opta.load(\n    f7_data = "",\n    f24_data = "OPTA Data/OPTA Data/F24 - Portugal/f24-99-2020-2133169-eventdetails.xml",\n    # Optional arguments\n    coordinates = "opta",\n    event_types = ["pass", "shot"]\n)\ndata_opta\n'

In [20]:
#df_opta = data_opta.to_df()
#df_opta.head()

In [21]:
#df_opta.info()

# <font color="#1c8a23">____________________________</font>
## <font color='#4b4b4b'>3. Comparação e Potencialidades</font> <a class="anchor" id="cp"></a>

Voltando aos dois exemplos de dados de eventos que obtivemos com sucesso (Statsbomb e Metrica Sports), vamos rever as colunas de cada um e o aspeto dos DataFrames.

In [22]:
print("Shape DataFrame Statsbomb:\n", df_statsbomb.shape)
print("Colunas DataFrame Statsbomb:\n", df_statsbomb.columns)
print("==========================================")
print("Shape DataFrame Metrica:\n", df_metrica.shape)
print("Colunas DataFrame Metrica:\n", df_metrica.columns)

Shape DataFrame Statsbomb:
 (1191, 21)
Colunas DataFrame Statsbomb:
 Index(['event_id', 'event_type', 'period_id', 'timestamp', 'end_timestamp',
       'ball_state', 'ball_owning_team', 'team_id', 'player_id',
       'coordinates_x', 'coordinates_y', 'end_coordinates_x',
       'end_coordinates_y', 'receiver_player_id', 'set_piece_type',
       'body_part_type', 'result', 'success', 'pass_type', 'is_under_pressure',
       'is_counter_attack'],
      dtype='object')
Shape DataFrame Metrica:
 (1493, 18)
Colunas DataFrame Metrica:
 Index(['event_id', 'event_type', 'period_id', 'timestamp', 'end_timestamp',
       'ball_state', 'ball_owning_team', 'team_id', 'player_id',
       'coordinates_x', 'coordinates_y', 'end_coordinates_x',
       'end_coordinates_y', 'receiver_player_id', 'set_piece_type', 'result',
       'success', 'body_part_type'],
      dtype='object')


In [23]:
display(df_statsbomb.head(), df_metrica.head())

,event_id,event_type,period_id,timestamp,end_timestamp,ball_state,ball_owning_team,team_id,player_id,coordinates_x,...,end_coordinates_x,end_coordinates_y,receiver_player_id,set_piece_type,body_part_type,result,success,pass_type,is_under_pressure,is_counter_attack
0,549567bd-36de-4ac8-b8dc-6b5d3f1e4be8,PASS,1,0 days 00:00:00.575000,0 days 00:00:02.590669,alive,206,206,6581,60.95,...,33.75,27.95,6855,KICK_OFF,LEFT_FOOT,COMPLETE,True,None,None,None
1,4e4e4cad-9897-43ec-842d-585a4077f6ce,PASS,1,0 days 00:00:03.864000,0 days 00:00:07.151421,alive,206,206,6855,36.75,...,86.45,74.15,None,None,RIGHT_FOOT,INCOMPLETE,False,LONG_BALL,None,None
2,be27cc25-92b5-4696-b43c-aad957a6119a,PASS,1,0 days 00:00:07.152000,0 days 00:00:08.848529,alive,217,217,5203,33.55,...,35.05,18.25,5470,None,None,COMPLETE,True,HIGH_PASS,True,None
3,b33c0b7f-7456-4efe-b43c-5fd7cbd14689,PASS,1,0 days 00:00:08.848000,0 days 00:00:09.983950,alive,217,217,5470,35.05,...,36.15,5.25,5477,None,HEAD,COMPLETE,True,HEAD_PASS,None,None
4,c587e5ce-fe6e-4cfb-b510-8a8e193699d3,PASS,1,0 days 00:00:10.873000,0 days 00:00:11.630764,alive,217,217,5477,34.25,...,25.25,1.55,5211,None,RIGHT_FOOT,COMPLETE,True,None,None,None


,event_id,event_type,period_id,timestamp,end_timestamp,ball_state,ball_owning_team,team_id,player_id,coordinates_x,coordinates_y,end_coordinates_x,end_coordinates_y,receiver_player_id,set_piece_type,result,success,body_part_type
0,2,PASS,1,0 days 00:00:14.400000,0 days 00:00:15.040000,alive,FIFATMA,FIFATMA,P3577,0.50125,0.48725,0.49864,0.48705,P3574,KICK_OFF,COMPLETE,True,None
1,4,PASS,1,0 days 00:00:15.320000,0 days 00:00:17,alive,FIFATMA,FIFATMA,P3574,0.49700,0.48500,0.63373,0.63449,P3575,None,COMPLETE,True,None
2,6,PASS,1,0 days 00:00:18.560000,0 days 00:00:20.240000,alive,FIFATMA,FIFATMA,P3575,0.66986,0.59707,0.80602,0.39821,P3569,None,COMPLETE,True,None
3,8,PASS,1,0 days 00:00:21.160000,0 days 00:00:23.160000,alive,FIFATMA,FIFATMA,P3569,0.80929,0.42922,0.79906,0.81522,P3570,None,COMPLETE,True,None
4,10,PASS,1,0 days 00:00:23.880000,0 days 00:00:25.080000,alive,FIFATMA,FIFATMA,P3570,0.79756,0.81998,0.68101,0.98059,P3571,None,COMPLETE,True,None


Como vemos, apesar de as colunas não serem exatamente as mesmas (nunca seriam, pelo facto de os diferentes provedores disponibilizarem diferentes tipos de dados), os nomes e conteúdo de cada uma são bastante homogéneos. E esta é, provavelmente, a maior vantagem do package kloppy (e o seu propósito, no fundo) - a partir daqui, podemos construir o nosso código sem nos preocuparmos em adaptá-lo para o formato de um determinado provedor. Tanto faz serem dados da Statsbomb, Metrica, OPTA, SkillCorner, Sportec, etc., que o código e a pipeline se mantêm os mesmos.

Para o demonstrar, e para terminar, vamos recorrer a uma outra função deste package, `event_pattern_matching` (que importámos acima com o alias "pm"). Esta função surge do conceito de [movement chains](https://www.statsperform.com/pt-br/resource/introducing-movement-chains/) proposto pela OPTA/StatsPerform e permite reconhecer a existência de certos padrões nos jogos das equipas, comparando-as e analisando as suas ocorrências dentro das partidas e evolução ao longo do tempo, por exemplo.

In [24]:
# Este é o exemplo apresentado na introdução da documentação (https://kloppy.pysport.org/user-guide/getting-started/#pattern-matching), apenas mudam as descrições para português.
# Aqui, vamos investigar sobre a ocorrência de sequências com (pelo menos) 4 passes bem sucedidos e que culminaram num remate
pattern = (
    # Todos os passes bem sucedidos...
    pm.match_pass(
        success = True,
        capture = "first_touch"
    )
    # ... a que se seguem outros 3 passes bem sucedidos por parte da mesma equipa...
    + pm.match_pass(
        success = True,
        team = pm.same_as("first_touch.team"),
    ) * 3
    # ... e que terminam num remate
    + pm.match_shot(
        team = pm.same_as("first_touch.team")
    )
)

Podemos criar uma função para procurar ocorrências destes eventos em qualquer conjunto de dados neste formato estandardizado.

In [25]:
def find_pattern(kloppy_data,
                 pattern):
  """
  A partir de um conjunto de dados de eventos (transformados pelo kloppy), pesquisa e devolve as ocorrências de uma sequência de eventos especificada.

  Parâmetros:
  - kloppy_data: Conjunto de dados de eventos
  - pattern: Sequência de eventos a procurar
  """

  search_pattern = pm.search(kloppy_data, pattern)
  print(f"Found {len(search_pattern)} matches\n")

  for match in search_pattern:
    print(" -> ".join([e.player.name for e in match.events]))

E agora aplicamos a função à partida e ao conjunto de dados da Statsbomb...

In [26]:
find_pattern(data_statsbomb, pattern)

Found 16 matches

Jordi Alba Ramos -> Marc-André ter Stegen -> Gerard Piqué Bernabéu -> Ivan Rakitić -> Lionel Andrés Messi Cuccittini
Jordi Alba Ramos -> Ousmane Dembélé -> Jordi Alba Ramos -> Luis Alberto Suárez Díaz -> Ousmane Dembélé
Ousmane Dembélé -> Ivan Rakitić -> Jordi Alba Ramos -> Ivan Rakitić -> Lionel Andrés Messi Cuccittini
Ivan Rakitić -> Sergi Roberto Carnicer -> Luis Alberto Suárez Díaz -> Lionel Andrés Messi Cuccittini -> Ousmane Dembélé
Nélson Cabral Semedo -> Gerard Piqué Bernabéu -> Ivan Rakitić -> Jordi Alba Ramos -> Luis Alberto Suárez Díaz
Ousmane Dembélé -> Luis Alberto Suárez Díaz -> Ousmane Dembélé -> Jordi Alba Ramos -> Ousmane Dembélé
Luis Alberto Suárez Díaz -> Ivan Rakitić -> Lionel Andrés Messi Cuccittini -> Ousmane Dembélé -> Jordi Alba Ramos
Gerard Piqué Bernabéu -> Sergio Busquets i Burgos -> Philippe Coutinho Correia -> Jordi Alba Ramos -> Philippe Coutinho Correia
Sergio Busquets i Burgos -> Lionel Andrés Messi Cuccittini -> Philippe Coutinho Correi

... e fazemos o mesmo para os dados da Metrica.

In [27]:
find_pattern(data_metrica, pattern)

Found 9 matches

Player 25 -> Player 19 -> Player 27 -> Player 22 -> Player 27
Player 25 -> Player 27 -> Player 26 -> Player 25 -> Player 27
Player 25 -> Player 21 -> Player 25 -> Player 27 -> Player 24
Player 24 -> Player 21 -> Player 22 -> Player 21 -> Player 27
Player 20 -> Player 21 -> Player 20 -> Player 27 -> Player 22
Player 4 -> Player 13 -> Player 1 -> Player 5 -> Player 1
Player 26 -> Player 25 -> Player 26 -> Player 25 -> Player 22
Player 22 -> Player 32 -> Player 21 -> Player 31 -> Player 32
Player 5 -> Player 14 -> Player 17 -> Player 12 -> Player 10


E cá está. Com a ajuda do kloppy, podemos estandardizar os dados de diferentes provedores e usar isso a nosso favor para efetuar análises, criar visualizações, simular cenários, entre outros, utilizando código comum e sem necessidade de estar a escrever linhas específicas para cada empresa fornecedora de dados.

Assim, podemos ocupar melhor o nosso tempo a construir soluções mais interessantes e avançadas, que permitam efetivamente responder às necessidades (ou curiosidades) de cada um.

Obrigado!